In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.4' 

%matplotlib widget

import jax
import matplotlib.pyplot as plt
import numpy as np

from temgym_core.components import Detector
from temgym_core.gaussian import MagneticPhaseSample, run_to_end, Lens, Biprism, circular_input_wave
from temgym_core.evaluate import evaluate_gaussians_gpu_kernel_wrapper
from temgym_core.transfer_matrices import calculate_z1_and_z2_from_M_and_f
from temgym_core.utils import energy2wavelength, reconstruct_with_background_removal

Simulation Parameters

In [ ]:
window_width = 1e-6  # metres
Nx = Ny = 512
pixel_size = window_width / Nx
voltage = 200e3  # volts
wavelength = energy2wavelength(voltage)
k = 2 * np.pi / wavelength

M1, f1 = -80, 1e-3 # Magnification and focal length of the lens
defocus = 0.0

Sample Creation and Visualization

In [ ]:
input_grid = Detector(
    z=0.0,
    pixel_size=(pixel_size, pixel_size),
    shape=(Ny, Nx),
)

sample = MagneticPhaseSample(
        z=input_grid.z,
        strength=0.8e-12,
        width=0.2e-6,
        height=0.2e-6,
        x0=0.25e-6,
        y0=0.05e-6,
        theta=-0.15,
        modulation_strength=0.002,
        skew_strength=0.01,
        radial_strength=0.5,
        edge_sharpness=1e8,
    )

sample_input_field = jax.vmap(sample.phase_shift)(input_grid.coords).reshape(input_grid.shape)

sample_input_field = np.exp(1j * k * sample_input_field)

extent = input_grid.extent
extent_um = tuple(float(val) * 1e6 for val in extent)
x_coords_um, y_coords_um = input_grid.coords_1d
x_coords_um = np.asarray(x_coords_um) * 1e6
y_coords_um = np.asarray(y_coords_um) * 1e6

In [28]:
title = 'Magnetic Sample Phase Shift'
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    np.angle(sample_input_field),
    extent=extent_um,
    cmap='grey',
)
ax.set_title(title)
ax.set_xlabel('x (\u00b5m)')
ax.set_ylabel('y (\u00b5m)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='Phase shift (rad)')

plt.show()


Input Beam and it's sample interaction visualised.

In [ ]:
rays_in = circular_input_wave(voltage=200e3, 
                                                    aperture_radius=window_width/2, 
                                                    num_rays=500_000,
                                                    waist=1e-9)

In [19]:
model = sample

run_to_end_jit = jax.jit(run_to_end)
run_to_end_vmapped = jax.jit(
    jax.vmap(run_to_end_jit, in_axes=(0, None)),
)
rays_out = run_to_end_vmapped(rays_in, (sample,))
field = evaluate_gaussians_gpu_kernel_wrapper(rays_out, input_grid).block_until_ready()

In [20]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im0 = axs[0].imshow(
    np.abs(field),
    extent=extent_um,
    origin='lower',
    cmap='inferno',
)
axs[0].set_title('Amplitude at Input')
axs[0].set_xlabel('x (\u00b5m)')
axs[0].set_ylabel('y (\u00b5m)')

im1 = axs[1].imshow(
    np.angle(field),
    extent=extent_um,
    cmap='hsv',
    vmin=-np.pi,
    vmax=np.pi,
)
axs[1].set_title('Phase at Input')
axs[1].set_xlabel('x (\u00b5m)')
axs[1].set_ylabel('y (\u00b5m)')

Text(0, 0.5, 'y (µm)')

Biprism Modelling

In [ ]:

z1, z2 = calculate_z1_and_z2_from_M_and_f(M=M1, f=f1) # object and image distances for a given magnification and focal length
z1, z2 = map(abs, (z1, z2))
lens = Lens(z=z1, focal_length=f1)
biprism = Biprism(z=(z1 + z2) / 2, strength=0.0004, width=1e-11) # Biprism deflection strength so the fringes line up well on the detector

# detector magnification in chosen to view interference pattern on the detector
detector_pixel_size = pixel_size * M1 * 0.6
detector = Detector(
    z=z1 + z2 + defocus,
    pixel_size=(detector_pixel_size, detector_pixel_size),
    shape=(Ny, Nx),
)
extent = detector.extent
extent_um = tuple(float(val) * 1e6 for val in extent)

Make Reference Hologram without Sample

In [22]:
rays_at_detector = run_to_end_vmapped(rays_in, (lens, biprism, detector))
reference_hologram = evaluate_gaussians_gpu_kernel_wrapper(rays_at_detector, detector).block_until_ready()
reference_hologram_intensity = np.abs(reference_hologram) ** 2

Make Hologram with Sample

In [23]:
rays_at_detector = run_to_end_vmapped(rays_in, (sample, lens, biprism, detector))
sample_hologram = evaluate_gaussians_gpu_kernel_wrapper(rays_at_detector, detector).block_until_ready()
sample_hologram_intensity = np.abs(sample_hologram) ** 2

Plot Reference Hologram and Sample Hologram

In [24]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im0 = axs[0].imshow(
    np.abs(reference_hologram),
    extent=extent_um,
    cmap='gray',
)
axs[0].set_title('Reference Hologram Intensity')
axs[0].set_xlabel('x (\u00b5m)')
axs[0].set_ylabel('y (\u00b5m)')

im1 = axs[1].imshow(
    np.abs(sample_hologram),
    extent=extent_um,
    cmap='gray',
)
axs[1].set_title('Sample Hologram Intensity')
axs[1].set_xlabel('x (\u00b5m)')
axs[1].set_ylabel('y (\u00b5m)')

Text(0, 0.5, 'y (µm)')

Reconstruct the sample hologram

In [25]:
out = reconstruct_with_background_removal(
    reference_hologram_intensity,
    sample_hologram_intensity,
    exclude_radius=5,
    sigma_pixels=5.0,
    detrend_plane=True,
)

# Final background-free phase map:
phi = out["background_removed"]["phase_final"]  # radians
amp_corr = out["background_removed"]["amp"]
mask = out["background_removed"]["mask"]
unwrapped = out["background_removed"]["phase_unwrapped"]

In [31]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
im0 = axs[0, 0].imshow(
    np.round(np.abs(sample_input_field)),
    extent=extent_um,
    cmap='gray',
)
axs[0, 0].set_title('Input Amplitude')
axs[0, 0].set_xlabel('x (\u00b5m)')
axs[0, 0].set_ylabel('y (\u00b5m)')

im1 = axs[0, 1].imshow(
    np.angle(sample_input_field),
    extent=extent_um,
    cmap='hsv',
    vmin=-np.pi,
    vmax=np.pi,
)
axs[0, 1].set_title('Input Phase')
axs[0, 1].set_xlabel('x (\u00b5m)')
axs[0, 1].set_ylabel('y (\u00b5m)')

im2 = axs[1, 0].imshow(
    amp_corr,
    extent=extent_um,
    cmap='gray',
)
axs[1, 0].set_title('Reconstructed Amplitude')
axs[1, 0].set_xlabel('x (\u00b5m)')
axs[1, 0].set_ylabel('y (\u00b5m)')

im3 = axs[1, 1].imshow(
    phi,
    extent=extent_um,
    cmap='hsv',
)
axs[1, 1].set_title('Reconstructed Phase')
axs[1, 1].set_xlabel('x (\u00b5m)')
axs[1, 1].set_ylabel('y (\u00b5m)')
plt.show()